In [2]:
# ==============================================================================
# PROYECTO: Financial Rates Auditor (Monitor de Divisas)
# DESCRIPCIÓN: Extracción automática de TRM para Dólar y Euro desde Google Finance.
# TECNOLOGÍA: Python + Requests + BeautifulSoup + Pandas
# AUTOR: [Tu Nombre / Portafolio]
# ==============================================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import os

# --- 1. CONFIGURACIÓN ESCALABLE ---
# Aquí puedes agregar todas las monedas que quieras monitorear fácilmente.
# Formato: 'Nombre': 'URL de Google Finance'
MONEDAS_OBJETIVO = {
    'Dolar (USD)': 'https://www.google.com/finance/quote/USD-COP',
    'Euro (EUR)':  'https://www.google.com/finance/quote/EUR-COP',
    # 'Libra (GBP)': 'https://www.google.com/finance/quote/GBP-COP', # (Ejemplo para escalar)
}

ARCHIVO_SALIDA = "reporte_divisas_pro.xlsx"

# Headers para simular ser un navegador real y evitar bloqueos (Robustez)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def obtener_tasa(nombre, url):
    """
    Conecta a la fuente de datos y extrae el valor numérico actual.
    Maneja errores de conexión o cambios en la estructura HTML.
    """
    try:
        print(f"🔄 Consultando: {nombre}...")
        respuesta = requests.get(url, headers=HEADERS)

        # Verificamos si la conexión fue exitosa (Código 200)
        if respuesta.status_code != 200:
            print(f"⚠️ Error de conexión con {nombre}: Código {respuesta.status_code}")
            return None

        soup = BeautifulSoup(respuesta.content, "html.parser")

        # BUSQUEDA DEL DATO (SELECTOR CSS)
        # Nota: La clase 'YMlKec fxKbKc' es la usada por Google Finance para el precio principal.
        elemento_precio = soup.find("div", class_="YMlKec fxKbKc")

        if elemento_precio:
            # Limpieza de datos: Quitamos comas y convertimos a decimal
            precio_texto = elemento_precio.text.replace(",", "")
            return float(precio_texto)
        else:
            print(f"⚠️ Alerta: No se encontró el selector de precio para {nombre}. La web pudo haber cambiado.")
            return None

    except Exception as e:
        print(f"❌ Error crítico en {nombre}: {e}")
        return None

def guardar_reporte(datos_recolectados):
    """
    Guarda o actualiza la base de datos en Excel sin sobrescribir el historial.
    """
    if not datos_recolectados:
        print("⚠️ No hay datos nuevos para guardar.")
        return

    # Creamos un DataFrame con los datos de hoy
    df_nuevo = pd.DataFrame(datos_recolectados)

    # Lógica de "Append" (Agregar al final)
    if os.path.exists(ARCHIVO_SALIDA):
        try:
            df_existente = pd.read_excel(ARCHIVO_SALIDA)
            df_final = pd.concat([df_existente, df_nuevo], ignore_index=True)
            print(f"📂 Archivo existente encontrado. Agregando {len(datos_recolectados)} registros nuevos.")
        except:
            df_final = df_nuevo # Si el archivo está corrupto, creamos uno nuevo
    else:
        df_final = df_nuevo
        print("📄 Creando nuevo archivo maestro.")

    # Guardar a Excel
    df_final.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"✅ Reporte actualizado exitosamente en: {ARCHIVO_SALIDA}")

# --- EJECUCIÓN DEL PROCESO ---
if __name__ == "__main__":
    resultados = []
    fecha_actual = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for moneda, url in MONEDAS_OBJETIVO.items():
        tasa = obtener_tasa(moneda, url)
        if tasa:
            resultados.append({
                'Fecha': fecha_actual,
                'Moneda': moneda,
                'Tasa': tasa
            })

    guardar_reporte(resultados)

🔄 Consultando: Dolar (USD)...
🔄 Consultando: Euro (EUR)...
📄 Creando nuevo archivo maestro.
✅ Reporte actualizado exitosamente en: reporte_divisas_pro.xlsx
